In [ ]:
! python3 -m pip install pynusinov

# EUVT/FUVT model use examples

In [ ]:
# (1) Importing the EUVT model into the current namespace
from pynusinov import Euvt2021

# (2) Creating an instance of the EUVT model
euvt = Euvt2021()

# (3) Calculating spectral bands and lines
bands, lines = euvt.get_spectra(lac=4.7)

In [ ]:
# (4) --
print(lines)

In [ ]:
# (5) --
print(bands)

In [ ]:
# (6) Importing modules required to preprocess and plot EUV-bands
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns; sns.set_theme()

# (7) Taking wavelength intervals and a flux as numpy arrays
lband = bands['lband'].to_numpy()
uband = bands['uband'].to_numpy()
euv_flux = bands['euv_flux_spectra'].to_numpy()

# (8) Converting the euv_flux array to 1D
#     for passing it to the plotting routine
euv_flux = euv_flux.flatten()

# (9) Making an array holding coordinates of all 'edges'
spectrum_edges = np.hstack([lband, uband[-1]])

# (10) --
plt.xlabel(r'Wavelength $\lambda$, $[nm]$')
plt.xticks(spectrum_edges, rotation=35)
plt.ylabel(r'Flux density $N_\lambda$ , $[m^{-2} \cdot s^{-1}]$')
plt.stairs(values=euv_flux, edges=spectrum_edges, fill=True)

In [ ]:
# (11) Importing the FUVT model into a current namespace
from pynusinov import Fuvt2021

# (12) Creating an instance of the FUVT model
fuvt = Fuvt2021()

# (13) Calculating FUV flux spectral bands
fuv_flux = fuvt.get_spectral_bands(lac=4.2)

# (14) --
print(fuv_flux)

# EUVN1992/XUVN1992 model use examples

In [ ]:
# (1) Importing the EUVN1992 model into the current namespace
from pynusinov import Euvn1992

# (2) Creating an instance of the EUVN1992 model
euvn = Euvn1992()

# (3) Calculating spectral bands and lines
bands, lines = euvn.get_spectra(hei=4.7)

In [ ]:
# (4) --
print(lines)

In [ ]:
# (5) --
print(bands)

In [ ]:
# (6) Importing modules required to preprocess and plot EUV-bands
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns; sns.set_theme()

# (7) Taking wavelength intervals and a flux as numpy arrays
lband = bands['lband'].to_numpy()
uband = bands['uband'].to_numpy()
euv_flux = bands['euv_flux_spectra'].to_numpy()

# (8) Converting the euv_flux array to 1D
#     for passing it to the plotting routine
euv_flux = euv_flux.flatten()

# (9) Making an array holding coordinates of all 'edges'
spectrum_edges = np.hstack([lband, uband[-1]])

# (10) --
plt.xlabel(r'Wavelength $\lambda$, $[nm]$')
plt.xticks(spectrum_edges, rotation=35)
plt.ylabel(r'Flux density $N_\lambda$ , $[m^{-2} \cdot s^{-1}]$')
plt.stairs(values=euv_flux, edges=spectrum_edges, fill=True)

In [ ]:
# (1) Importing the XUVN1992 model into the current namespace
from pynusinov import Xuvn1992

# (2) Creating an instance of the XUVN1992 model
xuvn = Xuvn1992()

# (3) Calculating spectral bands
bands = xuvn.get_spectra(i082=4.7)

In [ ]:
# (4) --
print(bands)

# Calculating He I from $F_{10.7}$ for Euvn1984 model

In [ ]:
from pynusinov import Euvn1984

# (1) Importing the Euvn1984 model into the current namespace
euvn = Euvn1984()

# (2) Predict He I value using the inner class HeI
hei = euvn.HeI.predict(f107=120, t=0.2)

# (3) --
print(hei)

In [ ]:
import pandas as pd
import numpy as np
import datetime
from matplotlib import pyplot as plt
from pynusinov import Euvn1984

# (1) Downloading AE-E and OMNI f10.7 satellite data and creating pandas datasets
!curl -O https://raw.githubusercontent.com/klklklo/pynusinov_misc/refs/heads/master/omni.csv
!curl -O https://raw.githubusercontent.com/klklklo/pynusinov_misc/refs/heads/master/aee.csv

hei_data = pd.read_csv('/content/aee.csv')
hei_data['time'] = pd.to_datetime(hei_data['time'])
f107 = pd.read_csv('/content/omni.csv')
f107['time'] = pd.to_datetime(f107['time'])
f107 = f107[(f107['time'] >= '1977-07-01') &
            (f107['time'] <= '1981-06-11')].reset_index(drop=True)

# (2) Creating a merged dataset aee_data from hei_data and f107
aee_data = pd.merge(hei_data, f107, on='time', how='inner')[['time', 'f107',
                                                             'hei_given']]

# (3) The beginning of the 21st solar cycle is March 1, 1976
st_date = datetime.datetime.strptime('1976-06-01', "%Y-%m-%d")

# (4) Calculation of He I using Nusinov's formula
hei_calc = []
euvn = Euvn1984()
for t_ in range(0, 771, 1):
    # (5) Converting the number of elapsed days into a part of the year
    t = (aee_data.iloc[t_]['time'] - st_date).days / 365
    hei_calc.append(Euvn1984.HeI.predict(aee_data.iloc[t_]['f107'], t)['hei'])

# (6) --
fig, ax = plt.subplots(1,1, figsize=(10, 6))
ax.scatter(aee_data['f107'], aee_data['hei_given'], s=7, color='red',
           label='AE-E data')
ax.scatter(aee_data['f107'], hei_calc, s=5, color='black',
           label='Nusinov\'s formula')
ax.set_xlabel(r'$\mathrm{F_{10.7}}$, s.f.u.', fontsize=14)
ax.set_ylabel(r'$\mathrm{He I,} 10^{13} \mathrm{photons} \cdot m^2 \cdot s$',
              fontsize=14)
ax.legend()
ax.grid()

In [ ]:
import pandas as pd
import numpy as np
import datetime
from matplotlib import pyplot as plt, ticker
from pynusinov import Euvn1984, Euvn1992

# (1) Downloading AE-E and OMNI f10.7 satellite data and creating pandas datasets
!curl -O https://raw.githubusercontent.com/klklklo/pynusinov_misc/refs/heads/master/omni.csv
!curl -O https://raw.githubusercontent.com/klklklo/pynusinov_misc/refs/heads/master/aee.csv

hei_data = pd.read_csv('/content/aee.csv')
hei_data['time'] = pd.to_datetime(hei_data['time'])
f107 = pd.read_csv('/content/omni.csv')
f107['time'] = pd.to_datetime(f107['time'])
f107 = f107[(f107['time'] >= '1977-07-01') &
            (f107['time'] <= '1981-06-11')].reset_index(drop=True)

# (2) Creating a merged dataset aee_data from hei_data and f107
aee_data = pd.merge(hei_data, f107, on='time', how='inner')[['time', 'f107',
                                                             'hei_given']]

# (3) The beginning of the 21st solar cycle is March 1, 1976
st_date = datetime.datetime.strptime('1976-06-01', "%Y-%m-%d")

# (4) Calculation of He I using Nusinov's formula
hei_1984_calc = []
for t_ in range(0, 771, 1):
    # (5) Converting the number of elapsed days into a part of the year
    t = (aee_data.iloc[t_]['time'] - st_date).days / 365
    hei_1984_calc.append(Euvn1984.HeI.predict(aee_data.iloc[t_]['f107'], t)['hei'])

# (6) --
fig, ax = plt.subplots(1,1, figsize=(11, 6))
ax.scatter(np.arange(771), aee_data['hei_given'], s=7, color='black',
           label='AE-E data')
ax.plot(hei_1984_calc, color='red', label='Nusinov\'s 1984 formula')
ax.xaxis.set_major_locator(ticker.MultipleLocator(100))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(20))
ax.set_xticks(np.arange(0, 800, 100),
             [aee_data.iloc[i, 0].strftime("%Y-%m-%d") for i in range(0,770,100)])
ax.set_ylabel(r'$\mathrm{He I,} 10^{13}  \mathrm{photons} \cdot m^2 \cdot s$',
              fontsize=14)
ax.legend()
ax.grid()

In [ ]:
import pandas as pd
import numpy as np
import datetime
from matplotlib import pyplot as plt, ticker
from pynusinov import Euvn1984, Euvn1992

# (1) Downloading AE-E and OMNI f10.7 satellite data and creating pandas datasets
!curl -O https://raw.githubusercontent.com/klklklo/pynusinov_misc/refs/heads/master/omni.csv
!curl -O https://raw.githubusercontent.com/klklklo/pynusinov_misc/refs/heads/master/aee.csv

hei_data = pd.read_csv('/content/aee.csv')
hei_data['time'] = pd.to_datetime(hei_data['time'])
f107 = pd.read_csv('/content/omni.csv')
f107['time'] = pd.to_datetime(f107['time'])
f107 = f107[(f107['time'] >= '1977-07-01') &
            (f107['time'] <= '1981-06-11')].reset_index(drop=True)

# (2) Creating a merged dataset aee_data from hei_data and f107
aee_data = pd.merge(hei_data, f107, on='time', how='inner')[['time', 'f107',
                                                             'hei_given']]

# (3) The beginning of the 21st solar cycle is March 1, 1976
st_date = datetime.datetime.strptime('1976-06-01', "%Y-%m-%d")

# (4) Calculation of He I using Nusinov's formula
hei_1984_calc = []
hei_1992_calc = []
for t_ in range(0, 771, 1):
    # (5) Converting the number of elapsed days into a part of the year
    t = (aee_data.iloc[t_]['time'] - st_date).days / 365
    hei_1984_calc.append(Euvn1984.HeI.predict(aee_data.iloc[t_]['f107'], t)['hei'])
    hei_1992_calc.append(Euvn1992.HeI.predict(aee_data.iloc[t_]['f107'], t)['hei'])

# (6) --
fig, ax = plt.subplots(1,1, figsize=(11, 6))
ax.scatter(np.arange(771), aee_data['hei_given'], s=7, color='black',
           label='AE-E data')
ax.plot(hei_1984_calc, color='red', label='Nusinov\'s 1984 formula')
ax.plot(hei_1992_calc, color='blue', label='Nusinov\'s 1992 formula')
ax.xaxis.set_major_locator(ticker.MultipleLocator(100))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(20))
ax.set_xticks(np.arange(0, 800, 100),
             [aee_data.iloc[i, 0].strftime("%Y-%m-%d") for i in range(0,770,100)])
ax.set_ylabel(r'$\mathrm{He I,} 10^{13}  \mathrm{photons} \cdot m^2 \cdot s$',
              fontsize=14)
ax.legend()
ax.grid()

# Calculating $I_{0.8-2.0}$ from $F_{10.7}$ for Xuvn1992 model

In [ ]:
from pynusinov import Xuvn1992

# (1) Importing the Xuvn1992 model into the current namespace
xuvn = Xuvn1992()

# (2) Predict I 0.8-2.0 value using the inner class I082
i082 = xuvn.I082.predict(f107=120)

# (3) --
print(i082)

In [ ]:
import pandas as pd
import numpy as np
import datetime
from matplotlib import pyplot as plt, ticker
from pynusinov import Xuvn1992

# (1) Downloading SOLRAD satellite data and creating pandas datasets
!curl -O https://raw.githubusercontent.com/klklklo/pynusinov_misc/refs/heads/master/xray_i082_f107.csv
solrad_data = pd.read_csv('/content/xray_i082_f107.csv')

# (2) Calculation of I 0.8-2.0 using Nusinov's formula
xuvn = Xuvn1992()
i082 = xuvn.I082.predict(f107=solrad_data['f107'])['i082']

# (3) --
fig, ax = plt.subplots(1,1, figsize=(10, 6))
ax.scatter(np.arange(solrad_data['i082'].size), solrad_data['i082'], s=5,
           color='black', label='SOLRAD 1978-01-01 - 1978-04-30')
ax.plot(i082, color='red', label='Nusinov\'s formula')
ax.xaxis.set_major_locator(ticker.MultipleLocator(15))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(5))
ax.set_xticks(np.arange(0, 121, 15),
              pd.date_range(start='1978-01-01',
                            end='1978-04-30', periods=9,).strftime("%Y-%m-%d"))
ax.set_ylabel(r'$\mathrm{I_{0.8-2.0}, W \cdot m^{-2}}$', fontsize=14)
ax.legend()
ax.grid()

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from pynusinov import Xuvn1992

# (1) Downloading SOLRAD satellite data and creating pandas datasets
!curl -O https://raw.githubusercontent.com/klklklo/pynusinov_misc/refs/heads/master/xray_i082_f107.csv
solrad_data = pd.read_csv('/content/pynusinov_misc/xray_i082_f107.csv')

# (2) Calculation of I 0.8-2.0 using Nusinov's formula
xuvn = Xuvn1992()
i082 = xuvn.I082.predict(f107=solrad_data['f107'])['i082']

# (3) --
fig, ax = plt.subplots(1,1, figsize=(10, 6))
ax.scatter(solrad_data['f107'], solrad_data['i082'], s=5, color='black',
           label='SOLRAD 1978-1983')
ax.plot(solrad_data['f107'], i082, color='red', label='Nusinov\'s formula')
ax.set_ylabel(r'$\mathrm{I_{0.8-2.0}, W \cdot m^{-2}}$', fontsize=14)
ax.set_xlabel(r'$\mathrm{F_{10.7}}$, s.f.u.', fontsize=14)
ax.legend()
ax.grid()